# Chapitre 3 : Exploration et nettoyage des données

*MLSMM2151 Data Science · Séance 2, vendredi 27 novembre, matin · Corentin Vande Kerckhove*

> **CONSEIL** : commencez par enregistrer une copie de ce notebook dans votre Google Drive
> (`Fichier` > `Enregistrer une copie dans Drive`) pour pouvoir prendre vos propres notes.

## Notebook 3 sur 3 : Variables temporelles

**Durée estimée : 10 minutes.**

### Objectifs

À la fin de ce notebook, vous saurez :

- convertir une colonne de dates stockées en texte, et en extraire une année ;
- calculer une durée par rapport à une date de référence fixe, et dire
  pourquoi fixe ;
- garder les lignes d'une période donnée, et savoir ce que le filtre écarte
  en silence.


In [ ]:
# Cette ligne récupère le fichier utils/donnees.py du dépôt du cours.
!curl -fsSL -o donnees.py https://raw.githubusercontent.com/cvandekerckh/mlsmm2151/main/utils/donnees.py

import pandas as pd
import matplotlib.pyplot as plt
from donnees import charger

In [ ]:
annonces = charger("airbnb_raw")

## 3.6. Traitement des variables temporelles

Une date stockée en texte se lit, mais ne se calcule pas : on ne peut pas
soustraire `"2010-08-11"` à `"2025-06-21"`. Les trois dates du fichier sont
dans ce cas. `format=` dit à pandas comment lire la date, au lieu de la
deviner : `"03/04/2025"` est le 3 avril pour un Belge et le 4 mars pour un
Américain.


In [ ]:
colonnes_dates = ["host_since", "first_review", "last_review"]
for colonne in colonnes_dates:
    annonces[colonne] = pd.to_datetime(annonces[colonne], format="%Y-%m-%d")

annonces[colonnes_dates].dtypes


Trois colonnes `datetime64`. Les manquants sont devenus `NaT`, *Not a Time*.
Une fois la colonne convertie, l'accesseur `.dt` donne accès à ses
composantes : `.dt.year`, `.dt.month`, `.dt.day_name()`. C'est l'équivalent,
pour les dates, de `.str` pour le texte.


In [ ]:
# En quelle année chaque annonce a-t-elle reçu son premier avis ?
annonces["first_review"].dt.year.value_counts().sort_index()

**Lecture, et piège.** Un creux en 2020, la pandémie, puis une forte hausse
jusqu'en 2024. Il serait tentant d'y lire la croissance d'Airbnb à Bruxelles.
Ce serait une erreur : ce fichier ne contient que les annonces **encore en
ligne le 21 juin 2025**. Les années anciennes sont mécaniquement
sous-représentées, on ne compte que les survivantes. C'est un **biais de
survie**, et il se cache dans toute photographie à une date donnée.


### Une durée, à partir d'une date fixe

Pour un modèle, une date brute est peu utile. Ce qui compte, c'est souvent une
**durée** : depuis combien de temps l'hôte est-il inscrit ? Une durée se
calcule par rapport à une date de référence. Nous prenons celle du fichier,
le 21 juin 2025.


In [ ]:
DATE_INSTANTANE = pd.Timestamp("2025-06-21")

In [ ]:
# Soustraire deux dates donne un écart en jours. On le convertit en années.
annonces["anciennete_hote_annees"] = (DATE_INSTANTANE - annonces["host_since"]).dt.days / 365.25
annonces["anciennete_hote_annees"].describe().round(2)

L'hôte médian est inscrit depuis 7,95 ans. On divise par 365,25 pour tenir
compte des années bissextiles. C'est exactement le calcul qui a produit la
colonne `anciennete_hote_annees` de la table `airbnb_train` du chapitre 5.

### Garder une période

Une date convertie se filtre comme un nombre. Gardons les annonces dont le
dernier avis date des douze mois qui précèdent le fichier : une définition
simple d'une annonce **active**.


In [ ]:
un_an_avant = DATE_INSTANTANE - pd.DateOffset(years=1)
actives = annonces[annonces["last_review"].between(un_an_avant, DATE_INSTANTANE)]
print(f"{len(actives)} annonces ont reçu un avis entre le {un_an_avant.date()} et le {DATE_INSTANTANE.date()}")

5 036 annonces sur 6 721. Le filtre a fait deux choses sans le dire : il a
écarté les 925 annonces sans avis, puisque `NaT` n'est dans aucun intervalle,
et les 31 annonces dont le dernier avis est postérieur au 21 juin, vues au
notebook 1. Un filtre sur une période est toujours aussi un filtre sur les
manquants.


### Hack-Time 🛠️

Nous venons de calculer l'ancienneté de l'hôte en années. Une annonce a, elle
aussi, un âge : le temps écoulé depuis son premier avis. À votre tour, dans la
cellule ci-dessous :

1. Créez la colonne `anciennete_annonce_annees`, à partir de `first_review` et
   de `DATE_INSTANTANE`. Une piste : reprenez la cellule de
   `anciennete_hote_annees` et changez une seule colonne.
2. Décrivez-la avec `describe()`, et comptez les annonces de moins d'un an.
3. Pourquoi y a-t-il 925 manquants, et que représentent-ils ?

**Quatre minutes.**

**Défi optionnel**
Comparez la médiane de `anciennete_annonce_annees` à celle de
`anciennete_hote_annees`. Les annonces sont nettement plus jeunes que leurs
hôtes : proposez deux explications, dont une qui tient au biais de survie.

In [ ]:
# Votre code ici

### Hack-Time 🛠️

Nous avons fixé `DATE_INSTANTANE` au 21 juin 2025. On aurait pu écrire
`pd.Timestamp.today()`, plus court. Dans la cellule ci-dessous :

1. Recalculez l'ancienneté de l'hôte avec `pd.Timestamp.today()` à la place
   de `DATE_INSTANTANE`, et comparez sa médiane à celle de
   `anciennete_hote_annees`.
2. Calculez `derniere_review_annees`, le temps écoulé entre le dernier avis et
   `DATE_INSTANTANE`, en années. Affichez son minimum et comptez les valeurs
   négatives.
3. En une phrase : pourquoi la date de référence doit-elle être écrite dans
   le code, et que faites-vous des valeurs négatives ?

**Trois minutes.**

**Défi optionnel**
Un modèle entraîné sur l'ancienneté calculée avec `today()` en novembre 2026
est utilisé en juin 2027. Qu'est-ce qui ne va pas ?


In [ ]:
# Votre code ici


## À retenir

- Une date en texte ne se calcule pas : `pd.to_datetime`, avec un `format=`
  explicite.
- `.dt` est à la date ce que `.str` est au texte : `.dt.year`,
  `.dt.day_name()`, et de nouvelles variables apparaissent.
- Une durée se calcule par rapport à une date de référence **fixe**, écrite
  dans le code. Avec `today()`, le notebook n'est plus reproductible.
- Un fichier photographié à une date ne contient que les survivants.
- Un filtre sur une période écarte aussi, en silence, les lignes sans date.

**La suite** : le devoir 1, que vous démarrez maintenant dans le notebook de
votre dépôt. Vous avez tous ses outils. Le chapitre 4, à la séance 3,
transformera ces colonnes nettoyées en variables qu'un modèle peut utiliser.
